**Data preprocessing** (run before) (code from "EDA(data cleaning)" file)

In [34]:
import pandas as pd

data = pd.read_csv("TrafficTwoMonth.csv")

# Convert traffic situation to numerical values
data['Traffic Situation'] = data['Traffic Situation'].map({
    'low': 0,
    'normal': 1,
    'high': 2,
    'heavy': 3
})

# Convert time to datetime
data['Time'] = pd.to_datetime(data['Time'])

# Extract useful features
data['Hour'] = data['Time'].dt.hour
data['Minute'] = data['Time'].dt.minute

# Drop original time column
data = data.drop(columns=['Time'])

# Encode day of week to int
data['Day of the week'] = data['Day of the week'].astype('category').cat.codes

# Feature selection
features = ['CarCount', 'BikeCount', 'BusCount', 'TruckCount', 'Total', 'Hour', 'Minute', 'Day of the week']
target = 'Traffic Situation'

X = data[features]
y = data[target]

# Normalisation
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Create sequences
import numpy as np

sequenceLength = 10  # number of time steps

X_sequences = []
y_sequences = []

for i in range(len(X_scaled) - sequenceLength):
    X_sequences.append(X_scaled[i:i+sequenceLength])
    y_sequences.append(y.iloc[i+sequenceLength])

X_sequences = np.array(X_sequences)
y_sequences = np.array(y_sequences)

/tmp/ipykernel_10471/3393176199.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Time'] = pd.to_datetime(data['Time'])


**Model Design and Implementation**

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [36]:
# Check the sequence data (testing)
print("Shape of X_sequences:", X_sequences.shape)
print("Shape of y_sequences:", y_sequences.shape)

print("\nFirst target values:")
print(y_sequences[:10])

Shape of X_sequences: (5942, 10, 8)
Shape of y_sequences: (5942,)

First target values:
[1 1 1 1 1 1 0 1 0 0]


In [37]:
# One-hot encode the target
y_encoded = to_categorical(y_sequences)

print("Shape of y_encoded:", y_encoded.shape)
print("\nFirst 5 encoded target rows:")
print(y_encoded[:5])

Shape of y_encoded: (5942, 4)

First 5 encoded target rows:
[[0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]]


In [38]:
# Train-test split
splitIndex = int(len(X_sequences) * 0.8)

X_train = X_sequences[:splitIndex]
X_test = X_sequences[splitIndex:]

y_train = y_encoded[:splitIndex]
y_test = y_encoded[splitIndex:]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (4753, 10, 8)
X_test shape: (1189, 10, 8)
y_train shape: (4753, 4)
y_test shape: (1189, 4)


In [39]:
# Handling Class Imbalance
y_trainLabels = np.argmax(y_train, axis=1)

classWeights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_trainLabels),
    y=y_trainLabels
)

classWeightsDict = dict(enumerate(classWeights))

print("Class weights:")
print(classWeightsDict)

Class weights:
{0: np.float64(1.8451086956521738), 1: np.float64(0.4014358108108108), 2: np.float64(4.069349315068493), 3: np.float64(1.3865227537922986)}


In [40]:
# Building RNN model
model = Sequential()

model.add(SimpleRNN(
    64,
    activation="tanh",
    input_shape=(X_train.shape[1], X_train.shape[2])
))

model.add(Dropout(0.2))

model.add(Dense(32, activation="relu"))
model.add(Dense(4, activation="softmax"))

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 64)             │         4,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,884 (26.89 KB)

 Trainable params: 6,884 (26.89 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
# Early stopping
earlyStopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True)

In [42]:
# Model training
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    class_weight=classWeightsDict,
    callbacks=[earlyStopping],
    verbose=1)

Epoch 1/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.4748 - loss: 1.1866 - val_accuracy: 0.4175 - val_loss: 1.2312
Epoch 2/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5287 - loss: 1.0480 - val_accuracy: 0.4690 - val_loss: 1.1194
Epoch 3/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5318 - loss: 0.9744 - val_accuracy: 0.4942 - val_loss: 1.0287
Epoch 4/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5247 - loss: 0.9453 - val_accuracy: 0.5163 - val_loss: 1.0097
Epoch 5/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5416 - loss: 0.9148 - val_accuracy: 0.5016 - val_loss: 0.9893
Epoch 6/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5381 - loss: 0.8915 - val_accuracy: 0.5089 - val_loss: 0.9930
Epoch 7/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5337 - loss: 0.9073 - val_accuracy: 0.5279 - val_loss: 0.9763
Epoch 8/30
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5466 - loss: 0.8679 - val_accuracy: 0